# Stockholm Bus Line 4 Spatial Data Pipeline

This notebook prepares the modelling dataset used in the project. It:

1. filters and quality-checks the January 2024 GTFS-RT data for Bus Line 4;
2. extracts the official Line 4 route shapes from static GTFS;
3. constructs directed stop-to-stop route segments;
4. derives road-infrastructure features from OpenStreetMap; and
5. produces the final master feature table shared by all prediction models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

RAW_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/raw'
PROCESSED_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed'

os.makedirs(PROCESSED_DIR, exist_ok=True)

RAW_FILE = os.path.join(RAW_DIR, 'gtfs_rt_202401.csv.gz')
OUTPUT_FILE = os.path.join(PROCESSED_DIR, 'line4_202401_clean.parquet')

# Quality-control fields provided in the historical GTFS-RT dataset.
QC_FIELD_1 = 'tripupdates_and_gtfs_not_merged_within_service_id'
QC_FIELD_2 = 'tripupdates_not_merged_with_stoptimes_or_trip'

print("Reading the January 2024 GTFS-RT dataset in chunks...")

chunks = []
total_rows = 0

for chunk in pd.read_csv(
    RAW_FILE,
    compression='gzip',
    sep=';',
    chunksize=500_000,
    low_memory=False
):
    total_rows += len(chunk)

    # Keep only observations successfully matched to the corresponding
    # static GTFS service, trip and stop-time information.
    valid_qc = (
        chunk[QC_FIELD_1]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(['f', 'false', '0'])
        &
        chunk[QC_FIELD_2]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(['f', 'false', '0'])
    )

    # Restrict the dataset to Stockholm Bus Line 4.
    valid_bus = (
        chunk['TransportMode']
        .astype(str)
        .str.upper()
        .eq('BUS')
    )

    valid_line4 = (
        chunk['LineNumber']
        .astype(str)
        .str.strip()
        .isin(['4', '04', '4.0'])
    )

    clean_chunk = chunk[
        valid_qc & valid_bus & valid_line4
    ]

    if not clean_chunk.empty:
        chunks.append(clean_chunk)

if not chunks:
    raise ValueError(
        "No valid Line 4 observations were found. "
        "Check the raw file and filtering fields."
    )

df_line4 = pd.concat(
    chunks,
    ignore_index=True
)

df_line4.to_parquet(
    OUTPUT_FILE,
    index=False
)

print(f"Raw observations scanned: {total_rows:,}")
print(f"Valid Line 4 observations: {len(df_line4):,}")
print(f"Saved to: {OUTPUT_FILE}")

## Read processed data file

In [ ]:
import os
import pandas as pd

PROCESSED_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed'
CLEAN_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_202401_clean.parquet'
)

df_line4 = pd.read_parquet(
    CLEAN_FILE
)

# The prediction target must be observed for an observation
# to be used in subsequent feature engineering.
df_trainable = df_line4.dropna(
    subset=['observed_arrival_delay']
).copy()

print("=== Clean Line 4 dataset ===")
print(f"Observations: {len(df_trainable):,}")
print(f"Unique trip IDs: {df_trainable['trip_id'].nunique():,}")
print(f"Service days: {df_trainable['date'].nunique()}")
print(
    "Date range:",
    df_trainable['date'].min(),
    "-",
    df_trainable['date'].max()
)

# Official GTFS Route Geometry

## Extract Representative Line 4 Shapes for Both Directions

In [ ]:
import os
import zipfile
import pandas as pd

PROCESSED_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed'

CLEAN_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_202401_clean.parquet'
)

STATIC_GTFS_ZIP = (
    '/content/drive/MyDrive/'
    'AH2173-Public Transport/'
    'Trafiklab_Data/'
    'sweden_gtfs.zip'
)

OUTPUT_SHAPE_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_shapes.parquet'
)

# ------------------------------------------------------------------
# 1. Identify the dominant official shape for each direction
# ------------------------------------------------------------------

df_clean = pd.read_parquet(
    CLEAN_FILE,
    columns=['direction_id', 'shape_id']
)

df_clean = df_clean.dropna(
    subset=['direction_id', 'shape_id']
).copy()

df_clean['direction_id'] = (
    df_clean['direction_id']
    .astype(int)
)

df_clean['shape_id'] = (
    df_clean['shape_id']
    .astype(str)
)

shape_counts = (
    df_clean
    .groupby(['direction_id', 'shape_id'])
    .size()
    .reset_index(name='observation_count')
)

dominant_shapes = (
    shape_counts
    .sort_values(
        ['direction_id', 'observation_count'],
        ascending=[True, False]
    )
    .groupby('direction_id')
    .head(1)
)

target_shapes = dict(
    zip(
        dominant_shapes['shape_id'],
        dominant_shapes['direction_id']
    )
)

print("Selected representative shapes:")
print(dominant_shapes)


# ------------------------------------------------------------------
# 2. Locate shapes.txt inside the Trafiklab static GTFS archive
# ------------------------------------------------------------------

with zipfile.ZipFile(
    STATIC_GTFS_ZIP,
    'r'
) as archive:

    shape_file = next(
        (
            name
            for name in archive.namelist()
            if name.endswith('shapes.txt')
            or name.endswith('shapes.csv')
        ),
        None
    )

    if shape_file is None:
        raise FileNotFoundError(
            "No shapes.txt or shapes.csv file was found "
            "inside the static GTFS archive."
        )


# ------------------------------------------------------------------
# 3. Stream the national shape file and retain the selected shapes
# ------------------------------------------------------------------

matched_chunks = []
total_scanned = 0

with zipfile.ZipFile(
    STATIC_GTFS_ZIP,
    'r'
) as archive:

    with archive.open(shape_file) as file_obj:

        for chunk in pd.read_csv(
            file_obj,
            chunksize=2_500_000,
            dtype={'shape_id': str},
            low_memory=False
        ):
            total_scanned += len(chunk)

            selected = chunk[
                chunk['shape_id'].isin(
                    target_shapes.keys()
                )
            ]

            if not selected.empty:
                matched_chunks.append(selected)

if not matched_chunks:
    raise ValueError(
        "The selected Line 4 shape IDs were not found "
        "in the static GTFS shape file."
    )


# ------------------------------------------------------------------
# 4. Prepare and save the two route geometries
# ------------------------------------------------------------------

df_shapes = pd.concat(
    matched_chunks,
    ignore_index=True
)

df_shapes['shape_pt_sequence'] = (
    df_shapes['shape_pt_sequence']
    .astype(int)
)

df_shapes['shape_pt_lat'] = (
    df_shapes['shape_pt_lat']
    .astype(float)
)

df_shapes['shape_pt_lon'] = (
    df_shapes['shape_pt_lon']
    .astype(float)
)

df_shapes['direction_id'] = (
    df_shapes['shape_id']
    .map(target_shapes)
    .astype(int)
)

df_shapes = (
    df_shapes
    .sort_values(
        ['direction_id', 'shape_pt_sequence']
    )
    .reset_index(drop=True)
)

df_shapes.to_parquet(
    OUTPUT_SHAPE_FILE,
    index=False
)

print(f"GTFS shape rows scanned: {total_scanned:,}")
print(f"Line 4 shape points retained: {len(df_shapes):,}")
print(
    df_shapes
    .groupby(['direction_id', 'shape_id'])
    .size()
    .reset_index(name='point_count')
)

print(f"Saved to: {OUTPUT_SHAPE_FILE}")

正在从 5500 万行轨迹中流式提取双向官方 Shape: ['1014010000631454799', '1014010000668999153'] ...
已扫描 12,500,000 行，命中双向轨迹点 821 个...
已扫描 25,000,000 行，命中双向轨迹点 821 个...
已扫描 37,500,000 行，命中双向轨迹点 821 个...
已扫描 50,000,000 行，命中双向轨迹点 821 个...

================ 双向官方轨迹提取完成 ================
文件已保存至: /content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/line4_shapes.parquet
   direction_id             shape_id  point_count
0             0  1014010000631454799          412
1             1  1014010000668999153          409


# OSM Spatial Feature Engineering

In [ ]:
!pip -q install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 2.0 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox

from shapely.geometry import LineString, Point
from shapely.ops import substring


# ------------------------------------------------------------------
# Paths and coordinate reference system
# ------------------------------------------------------------------

PROCESSED_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed'
RAW_STATIC_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/raw/static_gtfs'

CRS_METRIC = 'EPSG:3006'

SHAPE_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_shapes.parquet'
)

DYNAMIC_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_202401_clean.parquet'
)

STOPS_FILE = os.path.join(
    RAW_STATIC_DIR,
    'stops.csv'
)

SPATIAL_FEATURE_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_osm_segment_features.parquet'
)

MASTER_FEATURE_FILE = os.path.join(
    PROCESSED_DIR,
    'master_feature_table_line4.parquet'
)


# ------------------------------------------------------------------
# 1. Preserve relevant OSM bus-lane tags
# ------------------------------------------------------------------

bus_tags = [
    'busway',
    'busway:right',
    'busway:left',
    'busway:both',
    'lanes:bus',
    'lanes:bus:forward',
    'lanes:bus:backward',
    'bus:lanes',
    'bus:lanes:forward',
    'bus:lanes:backward',
    'psv',
    'psv:lanes',
    'lanes:psv',
    'lanes:psv:forward',
    'lanes:psv:backward'
]

ox.settings.useful_tags_way = list(
    set(
        ox.settings.useful_tags_way
        + bus_tags
    )
)

ox.settings.use_cache = True


# ------------------------------------------------------------------
# 2. Build the official route geometry for each direction
# ------------------------------------------------------------------

print("1/5 Building official Line 4 route geometries...")

df_shapes = pd.read_parquet(
    SHAPE_FILE
)

shape_dict = {}
shape_proj_list = []

for direction_id, group in df_shapes.groupby(
    'direction_id'
):

    group = group.sort_values(
        'shape_pt_sequence'
    )

    coordinates = list(
        zip(
            group['shape_pt_lon'],
            group['shape_pt_lat']
        )
    )

    line_wgs84 = LineString(
        coordinates
    )

    line_metric = (
        gpd.GeoSeries(
            [line_wgs84],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    shape_dict[int(direction_id)] = {
        'line_wgs84': line_wgs84,
        'line_metric': line_metric
    }

    shape_proj_list.append(
        line_metric
    )


# ------------------------------------------------------------------
# 3. Download the OSM road network and traffic signals
# ------------------------------------------------------------------

print("2/5 Downloading OSM road-network data...")

shape_series = gpd.GeoSeries(
    shape_proj_list,
    crs=CRS_METRIC
)

try:
    route_union = shape_series.union_all()
except AttributeError:
    route_union = shape_series.unary_union

corridor_metric = (
    route_union
    .buffer(200)
    .simplify(25)
)

corridor_wgs84 = (
    gpd.GeoSeries(
        [corridor_metric],
        crs=CRS_METRIC
    )
    .to_crs('EPSG:4326')
    .iloc[0]
)

graph = ox.graph_from_polygon(
    corridor_wgs84,
    network_type='drive'
)

_, edges = ox.graph_to_gdfs(
    graph
)

edges_metric = (
    edges
    .to_crs(CRS_METRIC)
    .copy()
)

try:
    signals = ox.features_from_polygon(
        corridor_wgs84,
        tags={'highway': 'traffic_signals'}
    )
except AttributeError:
    signals = ox.geometries_from_polygon(
        corridor_wgs84,
        tags={'highway': 'traffic_signals'}
    )

if signals.empty:
    signals_metric = gpd.GeoDataFrame(
        geometry=[],
        crs=CRS_METRIC
    )
else:
    signals_metric = (
        signals
        .to_crs(CRS_METRIC)
        .copy()
    )

    signals_metric['geometry'] = (
        signals_metric
        .geometry
        .centroid
    )


# ------------------------------------------------------------------
# 4. Identify road edges containing bus-lane attributes
# ------------------------------------------------------------------

def is_bus_edge(edge):
    """Return True when an OSM edge contains a bus-lane indicator."""

    target_prefixes = [
        'busway',
        'bus:lanes',
        'lanes:bus',
        'psv:lanes',
        'lanes:psv',
        'psv'
    ]

    for key, value in edge.items():

        if value is None:
            continue

        key_lower = str(key).lower()

        if not any(
            prefix in key_lower
            for prefix in target_prefixes
        ):
            continue

        if isinstance(
            value,
            (list, tuple, np.ndarray)
        ):
            values = list(value)
        else:
            values = [value]

        for item in values:

            if item is None:
                continue

            item_text = (
                str(item)
                .lower()
                .strip()
            )

            if item_text in [
                '',
                'nan',
                'none'
            ]:
                continue

            if any(
                keyword in item_text
                for keyword in [
                    'designated',
                    'lane',
                    'yes'
                ]
            ):
                return True

            if (
                item_text.isdigit()
                and int(item_text) > 0
            ):
                return True

    return False


edges_metric['is_bus'] = (
    edges_metric
    .apply(
        is_bus_edge,
        axis=1
    )
)

bus_edges_metric = (
    edges_metric[
        edges_metric['is_bus']
    ]
    .copy()
)

print(
    "OSM road edges with bus-lane attributes:",
    len(bus_edges_metric)
)


# ------------------------------------------------------------------
# 5. Construct directed stop-to-stop segments
# ------------------------------------------------------------------

print("3/5 Constructing directed stop-to-stop segments...")

try:
    stops = pd.read_csv(
        STOPS_FILE,
        sep=None,
        engine='python'
    )
except Exception:
    stops = pd.read_csv(
        STOPS_FILE,
        sep=';'
    )

stops['stop_id'] = (
    stops['stop_id']
    .astype(np.int64)
)

dynamic = pd.read_parquet(
    DYNAMIC_FILE
)

valid = dynamic.dropna(
    subset=[
        'observed_arrival_delay',
        'previous_stop_id',
        'observed_stop_id'
    ]
).copy()

valid['direction_id'] = (
    valid['direction_id']
    .astype(int)
)

valid['previous_stop_id'] = (
    valid['previous_stop_id']
    .astype(np.int64)
)

valid['observed_stop_id'] = (
    valid['observed_stop_id']
    .astype(np.int64)
)

segments = (
    valid[
        [
            'direction_id',
            'previous_stop_id',
            'observed_stop_id'
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

stop_attributes = stops[
    [
        'stop_id',
        'stop_name',
        'stop_lat',
        'stop_lon'
    ]
]

segments_with_coords = (
    segments
    .merge(
        stop_attributes.rename(
            columns={
                'stop_id': 'previous_stop_id',
                'stop_name': 'from_name',
                'stop_lat': 'from_lat',
                'stop_lon': 'from_lon'
            }
        ),
        on='previous_stop_id',
        how='inner'
    )
    .merge(
        stop_attributes.rename(
            columns={
                'stop_id': 'observed_stop_id',
                'stop_name': 'to_name',
                'stop_lat': 'to_lat',
                'stop_lon': 'to_lon'
            }
        ),
        on='observed_stop_id',
        how='inner'
    )
)


# ------------------------------------------------------------------
# 6. Calculate segment-level infrastructure features
# ------------------------------------------------------------------

segment_features = []

for _, row in segments_with_coords.iterrows():

    direction_id = int(
        row['direction_id']
    )

    if direction_id not in shape_dict:
        continue

    route_shape = (
        shape_dict[direction_id]
        ['line_metric']
    )

    from_point = Point(
        row['from_lon'],
        row['from_lat']
    )

    to_point = Point(
        row['to_lon'],
        row['to_lat']
    )

    from_point_metric = (
        gpd.GeoSeries(
            [from_point],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    to_point_metric = (
        gpd.GeoSeries(
            [to_point],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    from_distance = (
        route_shape
        .project(
            from_point_metric
        )
    )

    to_distance = (
        route_shape
        .project(
            to_point_metric
        )
    )

    if to_distance > from_distance:
        start_distance = from_distance
        end_distance = to_distance
    else:
        start_distance = min(
            from_distance,
            to_distance
        )
        end_distance = max(
            from_distance,
            to_distance
        ) + 1.0

    segment_metric = substring(
        route_shape,
        start_distance,
        end_distance
    )

    route_length_m = (
        segment_metric.length
    )

    segment_wgs84 = (
        gpd.GeoSeries(
            [segment_metric],
            crs=CRS_METRIC
        )
        .to_crs('EPSG:4326')
        .iloc[0]
    )

    corridor_15m = (
        segment_metric
        .buffer(15)
    )

    if signals_metric.empty:
        signal_count = 0
    else:
        signal_count = len(
            signals_metric[
                signals_metric
                .geometry
                .intersects(
                    corridor_15m
                )
            ]
        )

    matched_edges = edges_metric[
        edges_metric
        .geometry
        .intersects(
            corridor_15m
        )
    ]

    road_names = set()

    if 'name' in matched_edges.columns:

        for road_name in (
            matched_edges['name']
            .dropna()
        ):

            if isinstance(
                road_name,
                (list, tuple, np.ndarray)
            ):
                road_names.update(
                    list(road_name)
                )
            else:
                road_names.add(
                    road_name
                )

    intersection_count = max(
        1,
        len(road_names) - 1
    )

    if bus_edges_metric.empty:
        bus_lane_ratio = 0.0

    else:
        matched_bus_edges = (
            bus_edges_metric[
                bus_edges_metric
                .geometry
                .intersects(
                    corridor_15m
                )
            ]
        )

        if matched_bus_edges.empty:
            bus_lane_ratio = 0.0

        else:
            bus_length = (
                matched_bus_edges
                .geometry
                .intersection(
                    corridor_15m
                )
                .length
                .sum()
            )

            bus_lane_ratio = min(
                1.0,
                round(
                    bus_length
                    / max(
                        route_length_m,
                        1.0
                    ),
                    3
                )
            )

    segment_features.append(
        {
            'direction_id': direction_id,
            'previous_stop_id': row['previous_stop_id'],
            'observed_stop_id': row['observed_stop_id'],
            'from_name': row['from_name'],
            'to_name': row['to_name'],
            'network_length_m': round(route_length_m, 1),
            'num_traffic_signals': signal_count,
            'num_intersections': intersection_count,
            'bus_lane_ratio': bus_lane_ratio,
            'has_bus_priority': int(
                bus_lane_ratio >= 0.30
            ),
            'geometry_coords': [
                [lat, lon]
                for lon, lat
                in segment_wgs84.coords
            ]
        }
    )

spatial_features = pd.DataFrame(
    segment_features
)

spatial_features.drop(
    columns=['geometry_coords']
).to_parquet(
    SPATIAL_FEATURE_FILE,
    index=False
)

print(
    "\n=== Spatial feature summary ==="
)

print(
    spatial_features[
        [
            'network_length_m',
            'num_traffic_signals',
            'num_intersections',
            'bus_lane_ratio'
        ]
    ]
    .describe()
    .round(2)
)


# ------------------------------------------------------------------
# 7. Merge spatial features and construct temporal predictors
# ------------------------------------------------------------------

print("\n4/5 Building the master modelling table...")

JOIN_KEYS = [
    'direction_id',
    'previous_stop_id',
    'observed_stop_id'
]

master = valid.merge(
    spatial_features[
        JOIN_KEYS
        + [
            'network_length_m',
            'num_traffic_signals',
            'num_intersections',
            'bus_lane_ratio',
            'has_bus_priority'
        ]
    ],
    on=JOIN_KEYS,
    how='inner'
)

master = (
    master
    .sort_values(
        [
            'date',
            'trip_id',
            'direction_id',
            'stop_sequence'
        ]
    )
    .reset_index(drop=True)
)

group_keys = [
    'date',
    'trip_id',
    'direction_id'
]

master['delay_lag_1'] = (
    master
    .groupby(group_keys)
    ['observed_arrival_delay']
    .shift(1)
)

master['delay_lag_2'] = (
    master
    .groupby(group_keys)
    ['observed_arrival_delay']
    .shift(2)
)

master['delay_diff_prev'] = (
    master['delay_lag_1']
    - master['delay_lag_2']
)

master['arrival_hour'] = (
    master['scheduled_arrival_time_sfm']
    / 3600.0
)

master['hour_sin'] = np.sin(
    2 * np.pi
    * master['arrival_hour']
    / 24.0
)

master['hour_cos'] = np.cos(
    2 * np.pi
    * master['arrival_hour']
    / 24.0
)

# Peak-period indicators.
# Weekend status is represented separately by is_weekend.
master['is_morning_peak'] = (
    (
        master['arrival_hour'] >= 7.0
    )
    &
    (
        master['arrival_hour'] <= 9.0
    )
).astype(int)

master['is_evening_peak'] = (
    (
        master['arrival_hour'] >= 16.0
    )
    &
    (
        master['arrival_hour'] <= 18.5
    )
).astype(int)

master['date_dt'] = pd.to_datetime(
    master['date'].astype(str),
    format='%Y%m%d'
)

master['day_of_week'] = (
    master['date_dt']
    .dt
    .dayofweek
)

master['is_weekend'] = (
    master['day_of_week'] >= 5
).astype(int)

# At least one preceding stop is required for the prediction task.
master = (
    master
    .dropna(
        subset=['delay_lag_1']
    )
    .copy()
)

# For the second observation of a trip, lag 2 is not yet available.
# In this case, use lag 1 and define the previous change as zero.
master['delay_lag_2'] = (
    master['delay_lag_2']
    .fillna(
        master['delay_lag_1']
    )
)

master['delay_diff_prev'] = (
    master['delay_diff_prev']
    .fillna(0.0)
)

master.to_parquet(
    MASTER_FEATURE_FILE,
    index=False
)

print(
    f"Master table observations: {len(master):,}"
)

print(
    f"Directed segments: {len(spatial_features):,}"
)

print(
    f"Saved to: {MASTER_FEATURE_FILE}"
)

>>> 1/5: 正在构建 4 路双向官方 Shape 几何折线...
>>> 2/5: 正在下载走廊沿线 OSM 物理路网与红绿灯...
成功识别到走廊内包含专用道属性的道路边: 121 段！
>>> 3/5: 沿官方 Shape 裁切站间区间并计算微观属性...

================ 修正后核心特征描述性统计 ================
       network_length_m  num_traffic_signals  num_intersections  \
count             48.00                48.00              48.00   
mean             518.50                 5.58               4.27   
std              299.10                 3.34               2.56   
min              249.20                 0.00               1.00   
25%              324.50                 3.75               2.75   
50%              393.00                 5.50               4.00   
75%              592.83                 7.00               6.00   
max             1571.10                15.00              12.00   

       bus_lane_ratio  
count           48.00  
mean             0.66  
std              0.39  
min              0.00  
25%              0.26  
50%              0.86  
75%              1.00  
max              1.00 

In [ ]:
# ============================================================
# Cluster nearby OSM signal points into physical intersections
# ============================================================

import os
import pandas as pd
import numpy as np
import geopandas as gpd

from shapely.geometry import LineString, Point
from shapely.ops import substring


PROCESSED_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed'
RAW_STATIC_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project/data/raw/static_gtfs'

CRS_METRIC = 'EPSG:3006'

FEATURE_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_osm_segment_features.parquet'
)

MASTER_FILE = os.path.join(
    PROCESSED_DIR,
    'master_feature_table_line4.parquet'
)

SHAPE_FILE = os.path.join(
    PROCESSED_DIR,
    'line4_shapes.parquet'
)

STOPS_FILE = os.path.join(
    RAW_STATIC_DIR,
    'stops.csv'
)

print(
    "Clustering nearby OSM traffic-signal points "
    "into physical signalized intersections..."
)


# ------------------------------------------------------------------
# Load spatial features, route shapes and stops
# ------------------------------------------------------------------

df_features = pd.read_parquet(
    FEATURE_FILE
)

df_shapes = pd.read_parquet(
    SHAPE_FILE
)

try:
    stops_df = pd.read_csv(
        STOPS_FILE,
        sep=None,
        engine='python'
    )
except Exception:
    stops_df = pd.read_csv(
        STOPS_FILE,
        sep=';'
    )

stops_df['stop_id'] = (
    stops_df['stop_id']
    .astype(np.int64)
)

stop_lookup = (
    stops_df[
        [
            'stop_id',
            'stop_lat',
            'stop_lon'
        ]
    ]
    .drop_duplicates('stop_id')
    .set_index('stop_id')
)


# ------------------------------------------------------------------
# Rebuild the route shape for each direction
# ------------------------------------------------------------------

shape_dict = {}

for direction_id, group in df_shapes.groupby(
    'direction_id'
):

    group = group.sort_values(
        'shape_pt_sequence'
    )

    coordinates = list(
        zip(
            group['shape_pt_lon'],
            group['shape_pt_lat']
        )
    )

    shape_dict[int(direction_id)] = (
        gpd.GeoSeries(
            [LineString(coordinates)],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )


# ------------------------------------------------------------------
# Use the OSM signal layer already downloaded in the previous step
# ------------------------------------------------------------------

if 'signals_metric' in globals():

    signal_points = signals_metric.copy()

elif 'signals_proj' in globals():

    signal_points = signals_proj.copy()

else:
    raise RuntimeError(
        "The OSM signal layer is not available. "
        "Run the spatial-feature extraction cell first."
    )


# ------------------------------------------------------------------
# Recalculate signalized-intersection counts
# ------------------------------------------------------------------

clustered_counts = []

for _, row in df_features.iterrows():

    direction_id = int(
        row['direction_id']
    )

    from_id = int(
        row['previous_stop_id']
    )

    to_id = int(
        row['observed_stop_id']
    )

    route_shape = (
        shape_dict[direction_id]
    )

    from_stop = stop_lookup.loc[
        from_id
    ]

    to_stop = stop_lookup.loc[
        to_id
    ]

    from_point = Point(
        from_stop['stop_lon'],
        from_stop['stop_lat']
    )

    to_point = Point(
        to_stop['stop_lon'],
        to_stop['stop_lat']
    )

    from_point_metric = (
        gpd.GeoSeries(
            [from_point],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    to_point_metric = (
        gpd.GeoSeries(
            [to_point],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    from_distance = (
        route_shape
        .project(
            from_point_metric
        )
    )

    to_distance = (
        route_shape
        .project(
            to_point_metric
        )
    )

    start_distance = min(
        from_distance,
        to_distance
    )

    end_distance = max(
        from_distance,
        to_distance
    )

    segment = substring(
        route_shape,
        start_distance,
        end_distance
    )

    segment_corridor = (
        segment
        .buffer(15)
    )

    matched_signals = (
        signal_points[
            signal_points
            .geometry
            .intersects(
                segment_corridor
            )
        ]
    )

    if matched_signals.empty:

        clustered_count = 0

    else:

        signal_buffers = (
            matched_signals
            .geometry
            .buffer(18)
        )

        try:
            merged_signal_zones = (
                signal_buffers
                .union_all()
            )
        except AttributeError:
            merged_signal_zones = (
                signal_buffers
                .unary_union
            )

        if (
            merged_signal_zones
            .geom_type
            == 'Polygon'
        ):
            clustered_count = 1

        elif (
            merged_signal_zones
            .geom_type
            == 'MultiPolygon'
        ):
            clustered_count = len(
                merged_signal_zones.geoms
            )

        else:
            clustered_count = 1

    # Keep the signalized-intersection count physically
    # consistent with the total intersection count.
    clustered_count = min(
        clustered_count,
        int(
            row['num_intersections']
        )
    )

    clustered_counts.append(
        clustered_count
    )


df_features['num_traffic_signals'] = (
    clustered_counts
)

df_features.to_parquet(
    FEATURE_FILE,
    index=False
)

print(
    "\n=== Updated spatial feature summary ==="
)

print(
    df_features[
        [
            'num_traffic_signals',
            'num_intersections',
            'bus_lane_ratio'
        ]
    ]
    .describe()
    .round(2)
)


# ------------------------------------------------------------------
# Synchronize the corrected signal counts with the master table
# ------------------------------------------------------------------

master = pd.read_parquet(
    MASTER_FILE
)

master = master.drop(
    columns=['num_traffic_signals']
)

master = master.merge(
    df_features[
        [
            'direction_id',
            'previous_stop_id',
            'observed_stop_id',
            'num_traffic_signals'
        ]
    ],
    on=[
        'direction_id',
        'previous_stop_id',
        'observed_stop_id'
    ],
    how='left'
)

master.to_parquet(
    MASTER_FILE,
    index=False
)

print(
    "\nUpdated signal counts were synchronized "
    "with the master feature table."
)

## Final Interactive Infrastructure Map

In [ ]:
# ============================================================
# Standalone final interactive map
# Rebuild geometry from saved GTFS shape + stops
# No OSM re-download needed
# ============================================================

import os
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString
from shapely.ops import substring
import folium

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

BASE_DIR = '/content/drive/MyDrive/AH2179_AI_Transport_Project'

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    'data',
    'processed'
)

RAW_STATIC_DIR = os.path.join(
    BASE_DIR,
    'data',
    'raw',
    'static_gtfs'
)

FEATURE_PATH = os.path.join(
    PROCESSED_DIR,
    'line4_osm_segment_features.parquet'
)

SHAPE_PATH = os.path.join(
    PROCESSED_DIR,
    'line4_shapes.parquet'
)

STOPS_PATH = os.path.join(
    RAW_STATIC_DIR,
    'stops.csv'
)

CRS_METRIC = "EPSG:3006"

print("Loading saved files...")


# ------------------------------------------------------------
# 2. Load saved feature table
# ------------------------------------------------------------

df_features = pd.read_parquet(FEATURE_PATH)

df_shapes = pd.read_parquet(SHAPE_PATH)

try:
    stops_df = pd.read_csv(
        STOPS_PATH,
        sep=None,
        engine='python'
    )
except Exception:
    stops_df = pd.read_csv(
        STOPS_PATH,
        sep=';'
    )

stops_df['stop_id'] = stops_df['stop_id'].astype(np.int64)

df_features['previous_stop_id'] = (
    df_features['previous_stop_id'].astype(np.int64)
)

df_features['observed_stop_id'] = (
    df_features['observed_stop_id'].astype(np.int64)
)

print("Features:", df_features.shape)
print("Shapes:", df_shapes.shape)
print("Stops:", stops_df.shape)


# ------------------------------------------------------------
# 3. Build full GTFS shape geometry for each direction
# ------------------------------------------------------------

shape_dict = {}

for direction_id, group in df_shapes.groupby('direction_id'):

    group = group.sort_values('shape_pt_sequence')

    coords = list(
        zip(
            group['shape_pt_lon'],
            group['shape_pt_lat']
        )
    )

    line_wgs = LineString(coords)

    line_metric = (
        gpd.GeoSeries(
            [line_wgs],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    shape_dict[int(direction_id)] = line_metric

print("Shape directions:", shape_dict.keys())


# ------------------------------------------------------------
# 4. Prepare stop coordinate lookup
# ------------------------------------------------------------

stop_lookup = (
    stops_df[
        [
            'stop_id',
            'stop_name',
            'stop_lat',
            'stop_lon'
        ]
    ]
    .drop_duplicates('stop_id')
    .set_index('stop_id')
)


# ------------------------------------------------------------
# 5. Reconstruct true stop-to-stop shape geometry
# ------------------------------------------------------------

geometry_coords = []

for _, row in df_features.iterrows():

    direction_id = int(row['direction_id'])

    from_id = int(row['previous_stop_id'])
    to_id = int(row['observed_stop_id'])

    shape_metric = shape_dict[direction_id]

    from_stop = stop_lookup.loc[from_id]
    to_stop = stop_lookup.loc[to_id]

    from_point = Point(
        from_stop['stop_lon'],
        from_stop['stop_lat']
    )

    to_point = Point(
        to_stop['stop_lon'],
        to_stop['stop_lat']
    )

    from_point_metric = (
        gpd.GeoSeries(
            [from_point],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    to_point_metric = (
        gpd.GeoSeries(
            [to_point],
            crs='EPSG:4326'
        )
        .to_crs(CRS_METRIC)
        .iloc[0]
    )

    d_from = shape_metric.project(
        from_point_metric
    )

    d_to = shape_metric.project(
        to_point_metric
    )

    # Cut the official route shape between the two stops
    start_dist = min(d_from, d_to)
    end_dist = max(d_from, d_to)

    segment_metric = substring(
        shape_metric,
        start_dist,
        end_dist
    )

    # Convert back to WGS84 for Folium
    segment_wgs = (
        gpd.GeoSeries(
            [segment_metric],
            crs=CRS_METRIC
        )
        .to_crs('EPSG:4326')
        .iloc[0]
    )

    # Folium requires [lat, lon]
    coords_latlon = [
        [lat, lon]
        for lon, lat
        in segment_wgs.coords
    ]

    # Keep orientation consistent with stop order
    start_coord = coords_latlon[0]

    dist_to_from = (
        abs(start_coord[0] - from_stop['stop_lat'])
        +
        abs(start_coord[1] - from_stop['stop_lon'])
    )

    dist_to_to = (
        abs(start_coord[0] - to_stop['stop_lat'])
        +
        abs(start_coord[1] - to_stop['stop_lon'])
    )

    if dist_to_to < dist_to_from:
        coords_latlon = coords_latlon[::-1]

    geometry_coords.append(
        coords_latlon
    )


df_plot = df_features.copy()

df_plot['geometry_coords'] = (
    geometry_coords
)

print(
    "Reconstructed geometry for",
    len(df_plot),
    "segments"
)


# ------------------------------------------------------------
# 6. Create interactive map
# ------------------------------------------------------------

m = folium.Map(
    location=[59.343, 18.060],
    zoom_start=13,
    tiles=None
)

folium.TileLayer(
    tiles=(
        'https://server.arcgisonline.com/'
        'ArcGIS/rest/services/Canvas/'
        'World_Light_Gray_Base/'
        'MapServer/tile/{z}/{y}/{x}'
    ),
    attr='Esri',
    name='Base map'
).add_to(m)

# Separate directions
fg_out = folium.FeatureGroup(
    name='Outbound',
    show=True
)

fg_in = folium.FeatureGroup(
    name='Inbound',
    show=True
)


def get_color(bus_lane_ratio):

    if bus_lane_ratio >= 0.60:
        return '#2e8b57'     # high bus-lane coverage

    elif bus_lane_ratio >= 0.15:
        return '#e6a23c'     # partial coverage

    else:
        return '#d9534f'     # mixed traffic


# ------------------------------------------------------------
# 7. Draw route segments
# ------------------------------------------------------------

for _, row in df_plot.iterrows():

    ratio = float(
        row['bus_lane_ratio']
    )

    c = get_color(ratio)

    direction_name = (
        'Outbound'
        if row['direction_id'] == 0
        else 'Inbound'
    )

    popup_text = f"""
    <div style="
        font-family: Arial;
        min-width: 210px;
    ">
        <h4 style="
            margin:0 0 8px 0;
            color:#2c3e50;
        ">
            {row['from_name']} → {row['to_name']}
        </h4>

        <b>Direction:</b>
        {direction_name}<br>

        <b>Route segment length:</b>
        {row['network_length_m']:.0f} m<br>

        <b>Signalized intersections:</b>
        {int(row['num_traffic_signals'])}<br>

        <b>Total intersections:</b>
        {int(row['num_intersections'])}<br>

        <b>Bus lane ratio:</b>
        <span style="
            color:{c};
            font-weight:bold;
        ">
        {ratio*100:.1f}%
        </span>
    </div>
    """

    line = folium.PolyLine(
        locations=row[
            'geometry_coords'
        ],
        color=c,
        weight=5,
        opacity=0.9,
        popup=folium.Popup(
            popup_text,
            max_width=330
        ),
        tooltip=(
            f"{row['from_name']} → "
            f"{row['to_name']} | "
            f"Signalized intersections: "
            f"{int(row['num_traffic_signals'])} | "
            f"Bus lane ratio: "
            f"{ratio*100:.0f}%"
        )
    )

    if row['direction_id'] == 0:
        line.add_to(fg_out)

    else:
        line.add_to(fg_in)


# ------------------------------------------------------------
# 8. Add stops
# ------------------------------------------------------------

active_stops = (
    set(df_plot['previous_stop_id'])
    |
    set(df_plot['observed_stop_id'])
)

for stop_id in active_stops:

    stop = stop_lookup.loc[int(stop_id)]

    folium.CircleMarker(
        location=[
            stop['stop_lat'],
            stop['stop_lon']
        ],
        radius=3.5,
        color='#263238',
        weight=1.5,
        fill=True,
        fill_color='white',
        fill_opacity=1,
        tooltip=stop['stop_name']
    ).add_to(m)


# ------------------------------------------------------------
# 9. Add layers
# ------------------------------------------------------------

fg_out.add_to(m)
fg_in.add_to(m)

folium.LayerControl(
    collapsed=False
).add_to(m)


# ------------------------------------------------------------
# 10. English legend
# ------------------------------------------------------------

legend_html = """
<div style="
position: fixed;
bottom: 30px;
left: 30px;
width: 220px;
background-color: white;
border: 1px solid #999;
z-index: 9999;
padding: 12px;
font-size: 13px;
border-radius: 6px;
box-shadow: 1px 1px 5px rgba(0,0,0,0.15);
">

<b>Bus lane coverage</b><br><br>

<span style="
background:#2e8b57;
display:inline-block;
width:22px;
height:5px;
margin-right:8px;">
</span>
High (≥60%)<br>

<span style="
background:#e6a23c;
display:inline-block;
width:22px;
height:5px;
margin-right:8px;">
</span>
Partial (15–60%)<br>

<span style="
background:#d9534f;
display:inline-block;
width:22px;
height:5px;
margin-right:8px;">
</span>
Mixed traffic (<15%)<br><br>

○ Bus stop

</div>
"""

m.get_root().html.add_child(
    folium.Element(
        legend_html
    )
)


# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

HTML_PATH = os.path.join(
    PROCESSED_DIR,
    'line4_infrastructure_map_english.html'
)

m.save(
    HTML_PATH
)

print(
    "\nMap saved to:"
)

print(
    HTML_PATH
)

m

Loading saved files...
Features: (48, 10)
Shapes: (821, 6)
Stops: (21661, 7)
Shape directions: dict_keys([0, 1])
✅ Reconstructed geometry for 48 segments

✅ Map saved to:
/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/line4_infrastructure_map_english.html
